In [ ]:
import numpy as np
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import os

# ==========================================
# PHẦN 1: THUẬT TOÁN GAUSS XUẤT MARKDOWN
# ==========================================

def matrix_to_latex_core(np_mat, precision=4):
    """Hàm lõi tạo chuỗi bmatrix thuần (không chứa ký hiệu $$)"""
    mat = np.copy(np_mat).astype(float)
    mat[np.abs(mat) < 1e-10] = 0.0 # Xóa lỗi -0.0
    mat = np.round(mat, precision)
    
    latex_str = [r"\begin{bmatrix}"]
    for row in mat:
        # Dùng format %g để loại bỏ các số 0 thừa
        row_str = " & ".join([f"{val:g}" for val in row])
        latex_str.append("  " + row_str + r" \\")
    latex_str.append(r"\end{bmatrix}")
    return "\n".join(latex_str)

def matrix_to_latex(np_mat, precision=4):
    """Hàm bọc $$ cho ma trận đứng độc lập"""
    return "$$\n" + matrix_to_latex_core(np_mat, precision) + "\n$$"

def run_gauss(aug_mat, n_cols_b):
    m, total_cols = aug_mat.shape
    n_cols_a = total_cols - n_cols_b
    current_mat = aug_mat.astype(float).copy()
    
    md = ["# TRÌNH BÀY LỜI GIẢI KHỬ GAUSS (GIẢI TÍCH SỐ)\n"]
    md.append("### 1. Khởi tạo ma trận mở rộng $[A|B]$:\n")
    md.append(matrix_to_latex(current_mat) + "\n")
    
    pivot_list = [] # Lưu danh sách (hàng_i, cột_j)

    # ==========================================
    # GIAI ĐOẠN 1: QUÁ TRÌNH THUẬN (KHỬ)
    # ==========================================
    md.append("## GIAI ĐOẠN 1: QUÁ TRÌNH THUẬN (KHỬ XUÔI)\n")
    
    current_row = 0
    for j in range(n_cols_a):
        if current_row >= m: break
        
        # 1. Chọn phần tử khử aij (Ưu tiên từ trên xuống, trái sang)
        # Tìm hàng i >= current_row sao cho current_mat[i, j] != 0
        pivot_r = -1
        for i in range(current_row, m):
            if abs(current_mat[i, j]) > 1e-10:
                pivot_r = i
                break
        
        if pivot_r == -1: continue # Cột toàn 0, chuyển sang cột tiếp theo

        # Nếu phần tử khử không nằm ở hàng hiện tại, đổi chỗ hàng (để đảm bảo tính bậc thang)
        if pivot_r != current_row:
            current_mat[[pivot_r, current_row]] = current_mat[[current_row, pivot_r]]
            md.append(f"* Đổi chỗ hàng {pivot_r+1} và hàng {current_row+1}\n")
            pivot_r = current_row

        aij = current_mat[pivot_r, j]
        pivot_list.append((pivot_r, j))
        md.append(f"**Bước {len(pivot_list)}: Chọn phần tử khử $a_{{{pivot_r+1},{j+1}}} = {aij:g}$**\n")

        # 2. Tiến hành khử các hàng phía dưới (k từ i+1 đến m)
        for k in range(pivot_r + 1, m):
            akj = current_mat[k, j]
            if abs(akj) > 1e-10:
                factor = akj / aij
                current_mat[k] -= factor * current_mat[pivot_r]
                md.append(f"  * $L_{{{k+1}}} \\leftarrow L_{{{k+1}}} - ({akj:g}/{aij:g}) L_{{{pivot_r+1}}}$")
        
        md.append(matrix_to_latex(current_mat) + "\n")
        current_row += 1

    # ==========================================
    # GIAI ĐOẠN 2: QUÁ TRÌNH NGHỊCH (THẾ DẦN)
    # ==========================================
    md.append("## GIAI ĐOẠN 2: QUÁ TRÌNH NGHỊCH (THẾ DẦN)\n")
    
    # Kiểm tra vô nghiệm
    is_inconsistent = False
    for r in range(m):
        if r >= len(pivot_list): # Những hàng không chứa pivot
            if not np.allclose(current_mat[r, n_cols_a:], 0, atol=1e-10):
                is_inconsistent = True
                break
    
    if is_inconsistent:
        md.append("### Kết luận: Hệ phương trình VÔ NGHIỆM.\n")
        return "\n".join(md)

    # Để tìm nghiệm tổng quát (đặc biệt khi vô số nghiệm), 
    # ta đưa ma trận bậc thang về bậc thang rút gọn (RREF) bằng cách thế ngược
    for pr, pc in reversed(pivot_list):
        # Chuẩn hóa hàng pivot về 1
        val = current_mat[pr, pc]
        current_mat[pr] /= val
        
        # Khử ngược lên trên
        for k in range(pr):
            factor = current_mat[k, pc]
            current_mat[k] -= factor * current_mat[pr]

    md.append("### Kết quả sau khi thế dần (Ma trận RREF):\n")
    md.append(matrix_to_latex(current_mat) + "\n")

    # --- BIỆN LUẬN NGHIỆM (X0 và Vector tự do) ---
    pivot_cols = [p[1] for p in pivot_list]
    free_cols = sorted([c for c in range(n_cols_a) if c not in pivot_cols])
    
    X0 = np.zeros((n_cols_a, n_cols_b))
    for pr, pc in pivot_list:
        X0[pc, :] = current_mat[pr, n_cols_a:]

    if not free_cols:
        md.append("### Kết quả: Hệ có nghiệm duy nhất\n")
        md.append(matrix_to_latex(X0))
    else:
        free_str = ", ".join([f"x_{{{c+1}}}" for c in free_cols])
        md.append(f"### Kết quả: Hệ có vô số nghiệm, biến tự do: {{{free_str}}}\n")
        
        res_latex = "$$ X = " + matrix_to_latex_core(X0)
        for fc in free_cols:
            V = np.zeros((n_cols_a, 1))
            V[fc] = 1
            for pr, pc in pivot_list:
                V[pc] = -current_mat[pr, fc]
            
            res_latex += " + " + matrix_to_latex_core(V)
            if n_cols_b == 1:
                res_latex += f" t_{{{fc+1}}}"
            else:
                t_vars = " & ".join([f"t_{{{fc+1},{k+1}}}" for k in range(n_cols_b)])
                res_latex += r" \begin{bmatrix} " + t_vars + r" \end{bmatrix}"
        md.append(res_latex + " $$")

    return "\n".join(md)


# ==========================================
# PHẦN 2: GIAO DIỆN TKINTER NHẬP TỪ FILE
# ==========================================

class MatrixApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Công cụ Giải Gauss-Jordan (Hỗ trợ File txt)")
        self.root.geometry("600x500")
        
        config_frame = ttk.LabelFrame(root, text="1. Cấu hình vế phải", padding=10)
        config_frame.pack(fill="x", padx=10, pady=5)
        
        ttk.Label(config_frame, text="Số cột của ma trận B (vế phải):").grid(row=0, column=0, padx=5)
        self.cols_b_var = tk.IntVar(value=1)
        ttk.Entry(config_frame, textvariable=self.cols_b_var, width=5).grid(row=0, column=1)
        
        input_frame = ttk.LabelFrame(root, text="2. Nhập ma trận [A | B]", padding=10)
        input_frame.pack(fill="both", expand=True, padx=10, pady=5)
        
        toolbar = ttk.Frame(input_frame)
        toolbar.pack(fill="x", pady=(0, 5))
        
        ttk.Button(toolbar, text="Mở file Notepad (.txt)", command=self.load_file).pack(side="left")
        ttk.Label(toolbar, text=" Hoặc copy/paste thẳng vào ô bên dưới:", foreground="gray").pack(side="left", padx=10)
        
        self.text_area = tk.Text(input_frame, wrap="none", font=("Consolas", 11))
        self.text_area.pack(fill="both", expand=True)
        
        sample_data = "1 2 3 4\n5 6 7 8\n9 10 11 12"
        self.text_area.insert("1.0", sample_data)
        
        btn_frame = ttk.Frame(root, padding=10)
        btn_frame.pack(fill="x")
        ttk.Button(btn_frame, text="GIẢI & XUẤT FILE MARKDOWN", command=self.solve, style="Accent.TButton").pack(pady=10)

    def load_file(self):
        filepath = filedialog.askopenfilename(filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")])
        if filepath:
            with open(filepath, 'r') as f:
                content = f.read()
            self.text_area.delete("1.0", tk.END)
            self.text_area.insert(tk.END, content)

    def solve(self):
        try:
            n_cols_b = self.cols_b_var.get()
            if n_cols_b <= 0:
                raise ValueError
        except:
            messagebox.showerror("Lỗi", "Số cột của B phải là số nguyên > 0!")
            return
            
        raw_text = self.text_area.get("1.0", tk.END).strip()
        if not raw_text:
            messagebox.showerror("Lỗi", "Dữ liệu ma trận đang trống!")
            return
            
        try:
            import io
            aug_mat = np.loadtxt(io.StringIO(raw_text))
            if aug_mat.ndim == 1: 
                aug_mat = aug_mat.reshape(1, -1)
        except Exception as e:
            messagebox.showerror("Lỗi Cú Pháp", "Không thể đọc ma trận. Hãy đảm bảo dữ liệu chỉ chứa số và được phân cách bằng dấu cách hoặc tab.\nChi tiết: " + str(e))
            return
            
        m, total_cols = aug_mat.shape
        if total_cols <= n_cols_b:
            messagebox.showerror("Lỗi Kích Thước", f"Tổng số cột đọc được ({total_cols}) phải lớn hơn số cột của B ({n_cols_b})!")
            return
            
        save_path = filedialog.asksaveasfilename(
            defaultextension=".md",
            filetypes=[("Markdown Files", "*.md")],
            title="Lưu lời giải vào file"
        )
        
        if not save_path:
            return 
            
        print("\n[HỆ THỐNG] Đang giải và tạo file Markdown...")
        md_content = run_gauss(aug_mat, n_cols_b)
        
        with open(save_path, 'w', encoding='utf-8') as f:
            f.write(md_content)
            
        messagebox.showinfo("Thành công", f"Đã xuất lời giải siêu đẹp ra file:\n{save_path}")

if __name__ == "__main__":
    root = tk.Tk()
    style = ttk.Style()
    style.configure("Accent.TButton", font=("Arial", 10, "bold"), foreground="black")
    app = MatrixApp(root)
    root.mainloop()


[HỆ THỐNG] Đang giải và tạo file Markdown...
